# Chapter 1: What Data Science Is, and How We Work

**Foundations of Data Science**

## In this lesson

1. What a data scientist actually does (and what this course is and isn't)
2. The workflow we'll use all semester: question, data, exploration, model, decision
3. Python and pandas as the working environment
4. First look at a real dataset: NHANES, the US national health survey
5. The course project brief

## The one idea

A data science problem starts with a **decision someone has to make**, not with a dataset or a model. Every chapter we'll open with the decision, then ask what data and what method would inform it. If you can't say who makes the decision and what changes depending on the answer, you don't have a data science problem yet.

## Setup

The cell below works in Google Colab or a local Jupyter session. It clones the course repo, installs requirements, and fetches the data this notebook needs. Run it once per session.

<!-- nb-links -->
[Download this notebook](chapter_01_intro.ipynb?download=1){download="chapter_01_intro.ipynb"} &nbsp;|&nbsp; [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lshpaner/fds_course_students/blob/main/notebooks/chapter_01/chapter_01_intro.ipynb)


In [ ]:
# --- Setup: works in Colab and locally ---------------------------------------
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "fds_course"

if IN_COLAB:
    # Clone the course repo and move into it (skip if already there)
    if not Path(REPO).exists():
        !git clone -q https://github.com/lshpaner/{REPO}.git
    os.chdir(REPO)
    !pip install -q -r requirements.txt
else:
    # Local: the notebook lives in notebooks/chapter_XX/, so go up two levels to the repo root
    if Path.cwd().name.startswith("chapter_"):
        os.chdir(Path.cwd().parents[1])

# Make core/ importable
sys.path.insert(0, os.getcwd())

# Fetch this chapter's raw data only if it isn't already on disk, then build the processed table
for component in ["nhanes_demo", "nhanes_bmx", "nhanes_bpx", "nhanes_diq"]:
    if not Path(f"data/raw/{component}.parquet").exists():
        !python data_gen.py fetch --name {component}
if not Path("data/processed/nhanes.parquet").exists():
    !python preprocessing.py

print("Working directory:", os.getcwd())

## 1. Start with the decision, not the data

**NHANES** (National Health and Nutrition Examination Survey) is run by the CDC. Every two years about 10,000 people across the US are interviewed at home and then examined in a mobile clinic: body measurements, blood pressure, lab work, questionnaires. It's public, it's real, and it has every problem real data has. We'll use the 2017-2018 cycle.

The decision for today. In 2021 the US Preventive Services Task Force lowered the recommended starting age for diabetes screening from 40 to 35 for adults who are overweight or obese. A primary-care clinic is deciding whether to go further and screen *all* adults from 35, regardless of weight.

> **Who decides, and what changes depending on the answer?** The clinic's medical director decides. If diabetes is rare in normal-weight 35-year-olds, universal screening wastes visits and money and generates false positives. If it isn't rare, the weight cutoff misses people.

So the question for the data: **how does diabetes prevalence change with age, and does weight change that picture?** Keep it in mind as we look at the table.

## 2. First contact with the data

Three things to do with any new table, in this order:

1. **Shape and types**: how many rows and columns, what kind of value is in each
2. **A few rows**: does it look like what the documentation says?
3. **Missingness**: what's absent, and is it absent at random?

`preprocessing.py` already merged four NHANES files into one table and gave the columns readable names. Next chapter you'll see exactly how; today we take the table as given. We never read from the download URL inside a notebook; the file on disk is the single source of truth for the semester.

In [ ]:
import pandas as pd

# Read the merged table written by preprocessing.py
nhanes = pd.read_parquet("data/processed/nhanes.parquet")

# 1. Shape and types
print("shape:", nhanes.shape)
nhanes.info()

In [ ]:
# 2. A few rows. Codes to know: sex 1 = male, 2 = female; race_eth is a CDC category code;
#    income_ratio is family income divided by the poverty line (capped at 5); weight_mec is the survey weight.
nhanes.head()

In [ ]:
# 3. Missingness: count per column, sorted. Blood pressure is the big one; ask why before you do anything about it.
nhanes.isna().sum().sort_values(ascending=False)

### The one-liner you'll use from here on

Those three calls are what every analyst runs first. `eda_toolkit` wraps them into one table: dtype, null count and percent, unique count, and the most common value per column. Use it whenever you meet a new table; the point of seeing the three pandas calls first is that you know what it's doing.

In [ ]:
from eda_toolkit import dataframe_profiler

# One table: column, dtype, nulls, unique values, most frequent value and its share
dataframe_profiler(nhanes, return_df=True)

### The first decision about the data

Check the age range. NHANES interviews everyone from infancy up, and the diabetes question is asked from age 1. Blood pressure isn't measured under age 8. Our clinic question is about adults, so the honest move is to **restrict to age 20 and over** before summarizing anything. Every filter like this is a decision; write it down and say why.

(A second thing to know and set aside for now: `weight_mec` is a survey weight. Each row stands in for tens of thousands of Americans, and the right way to estimate a national prevalence uses those weights. We'll come back to that in chapter 3. Today's unweighted numbers are fine for seeing the shape of the problem.)

In [ ]:
# Restrict to adults. Keep the filter in one named variable so it is easy to find and change.
adults = nhanes[nhanes["age"] >= 20].copy()
print("adults:", adults.shape[0], "of", nhanes.shape[0], "rows")
adults["age"].describe().round(1)

## 3. Summarize before you model

A grouped summary table is the cheapest model there is. If a plain `groupby` already answers the question, a fancy model would only confirm it.

Two steps: how common is diabetes overall, and how does it move with age?

In [ ]:
# How common is diabetes among adults? value_counts(normalize=True) gives proportions.
adults["diabetes"].value_counts(normalize=True).round(3)

In [ ]:
# Prevalence by 10-year age band. pd.cut turns a number into labeled bins; the mean of a 0/1 column is a proportion.
age_bins = [20, 30, 40, 50, 60, 70, 81]
age_labels = ["20-29", "30-39", "40-49", "50-59", "60-69", "70+"]
adults["age_band"] = pd.cut(adults["age"], bins=age_bins, labels=age_labels, right=False)

prev_by_age = adults.groupby("age_band", observed=True)["diabetes"].agg(["mean", "count"]).round(3)
prev_by_age.columns = ["prevalence", "n"]
prev_by_age

## 4. One picture

A table tells you *that* prevalence rises with age. A plot tells you *whether weight changes the picture*, which is the clinic's actual question. We'll spend all of chapter 5 on visualization; for now, one line plot: prevalence by age band, one line per BMI category.

BMI categories (CDC): under 25 normal, 25 to 30 overweight, 30 and over obese.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# BMI category. Rows with missing BMI fall out of this plot; note how many.
adults["bmi_cat"] = pd.cut(
    adults["bmi"], bins=[0, 25, 30, 100], labels=["normal", "overweight", "obese"], right=False
)
print("missing BMI among adults:", adults["bmi"].isna().sum())

# Prevalence by age band and BMI category
prev = (
    adults.dropna(subset=["bmi_cat"])
    .groupby(["age_band", "bmi_cat"], observed=True)["diabetes"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.lineplot(data=prev, x="age_band", y="diabetes", hue="bmi_cat", marker="o", ax=ax)
ax.axvline(1.5, color="gray", linestyle="--", linewidth=1)  # between 30-39 and 40-49: the age-35 line, roughly
ax.set_xlabel("Age band")
ax.set_ylabel("Share reporting diabetes")
ax.set_title("Diabetes prevalence rises with age in every weight group, but not equally")
ax.legend(title="BMI category")
plt.show()

### Back to the decision

Read the plot against the clinic's question. In the 30-39 band, prevalence in the normal-weight group is under 2%, while the obese group is already about three times that. By 50-59 it is roughly 6% against 24%. So the data say: a weight-based cutoff at 35 is catching most of the risk, and universal screening at 35 would mostly be testing people with very low prevalence. The medical director now has a number to weigh against the cost of a missed case, which is a judgment call the data can't make for them.

Notice what we did *not* do: fit a model. We'll fit one in chapter 7, on this same table, and it will mostly confirm what this plot already says. That is the normal order of operations: question, data, summary, picture, and only then a model, if the decision still needs one.

Also notice what we glossed over, each of which gets its own chapter: how the four NHANES files were joined (chapter 2), what to do about the 900 missing BMIs and 2,600 missing blood pressures (chapter 2), whether "reports diabetes" is the same as "has diabetes" (chapter 3 and chapter 13), and the survey weights (chapter 3).

## 5. What this course is, and isn't

The syllabus says it plainly: this is a course about learning to think with data, not a course in algorithms or a tour of a library. Each chapter follows the loop we just walked through:

| Step | What you do | Where in the course |
|---|---|---|
| Question | Name the decision and who makes it | Every chapter, first section |
| Data | Get it, handle what's missing, know its limits | Chapters 2, 3 |
| Explore | Summaries and pictures before models | Chapters 4, 5 |
| Model | Regression, classification, clustering, time series, networks, language models | Chapters 6 through 12 |
| Evaluate | Would you act on this? What does it cost to be wrong? | Chapter 8, and every chapter after |
| Decide and communicate | Say what you'd do, and why, to someone who didn't see the code | Chapters 13, 14, and the project |

Tools: Python, pandas, scikit-learn, statsmodels, matplotlib and seaborn, plus a few open-source libraries I maintain (`eda_toolkit`, `model_metrics`, `model_tuner`, `equiboots`) where they save us time. If you've never used pandas, read VanderPlas chapter 3 before the next one; everything else we'll pick up as we go.

## 6. The course project

Teams of three. You choose a problem where someone has a real decision to make, find public data that bears on it, and work the loop above end to end.

**Three deliverables:**

1. **Problem and background** (chapter 4): one page. Who decides, what changes depending on the answer, what data exists, what's already been tried.
2. **Proposal pitch** (chapter 6): five-minute presentation. The question, the data you've actually loaded, your first summary table or plot, and your plan.
3. **Implementation and presentation** (chapter 14): notebook plus a 12-minute talk. Graded on reasoning and communication, not on the fanciest model.

**Example problems** (you may pick one of these or bring your own):

- Which neighborhoods should a city prioritize for traffic-calming, given collision records? (open city data portals: [LA](https://data.lacity.org/), [NYC](https://data.cityofnewyork.us/Public-Safety/Motor-Vehicle-Collisions-Crashes/h9gi-nx95), [Chicago](https://data.cityofchicago.org/))
- Can a hospital predict which patients will miss appointments, and what would it do with that? ([public no-show dataset](https://www.kaggle.com/datasets/joniarroba/noshowappointments))
- Which products should a retailer bundle, based on transaction data? ([UCI Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail))
- Where should a bike-share add docks, based on trip and station data? ([Citi Bike](https://citibikenyc.com/system-data), [Divvy](https://divvybikes.com/system-data), [Metro Bike Share](https://bikeshare.metro.net/about/data/))
- Which factors predict whether a Kickstarter campaign funds? ([Kaggle](https://www.kaggle.com/datasets/kemical/kickstarter-projects))
- Is air quality in a city getting better or worse, and does it depend on where you live? ([EPA AirNow](https://www.airnow.gov/), [OpenAQ](https://openaq.org/))

A problem with no decision attached ("predict X because the data is there") will be sent back.

## 7. Looking ahead

- **Reading**: VanderPlas, *Python Data Science Handbook*, chapter 3 (pandas). ISLR chapter 1 and 2.1 for the big picture.
- **Project**: start forming teams and talking about problems; the one-page statement is due in chapter 4.

## Appendix: reproducing this notebook

Everything here runs from the raw Parquet file written by `data_gen.py`. If you change the data on disk, re-run `make data_gen` and start the notebook from the top. We'll keep that discipline all semester: raw data is fetched by a script, never edited by hand, and never committed to git.